In [2]:
# Core libraries
import pandas as pd
import numpy as np
import os

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# PyCaret
from pycaret.regression import *

# Model saving
import joblib

print("PyCaret Regression Notebook Initialized")

PyCaret Regression Notebook Initialized


In [3]:
# Path to processed dataset
PROCESSED_DATA_PATH = './../../data/electricity/processed/demand_model_ready.parquet'
MODEL_PATH = './../../data/trained_models/electricity/'

# Load dataset
df = pd.read_parquet(PROCESSED_DATA_PATH)
print(f"Loaded {len(df):,} records")

# Ensure datetime
df['SETTLEMENT_DATE'] = pd.to_datetime(df['SETTLEMENT_DATE'])

# Sort chronologically for time-aware splitting
df = df.sort_values('SETTLEMENT_DATE').reset_index(drop=True)

Loaded 434,590 records


In [4]:
# Calendar features
df['Month'] = df['SETTLEMENT_DATE'].dt.month
df['Day'] = df['SETTLEMENT_DATE'].dt.day
df['DayOfWeek'] = df['SETTLEMENT_DATE'].dt.dayofweek
df['Year'] = df['SETTLEMENT_DATE'].dt.year

# Target
df['log_demand'] = np.log(df['ND'])

# Features for PyCaret
features = ['Month', 'Day', 'DayOfWeek', 'SETTLEMENT_PERIOD', 'Year']

In [5]:
# 80/20 chronological split
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

print(f"Training samples: {len(train_df):,}")
print(f"Testing samples : {len(test_df):,}")

Training samples: 347,672
Testing samples : 86,918


In [6]:
print(features)

['Month', 'Day', 'DayOfWeek', 'SETTLEMENT_PERIOD', 'Year']


In [19]:
reg = setup(
    data=train_df,
    target='log_demand',
    numeric_features=features,
    ignore_features=[col for col in train_df.columns if col not in features + ['log_demand']],
    session_id=42,
    fold_strategy='timeseries',
    fold=5,
    data_split_shuffle=False,
    fold_shuffle=False
)

print(get_config('X_train').columns)

,Description,Value
0,Session id,42
1,Target,log_demand
2,Target type,Regression
3,Original data shape,"(347672, 27)"
4,Transformed data shape,"(347672, 6)"
5,Transformed train set shape,"(243370, 6)"
6,Transformed test set shape,"(104302, 6)"
7,Ignore features,21
8,Numeric features,5
9,Categorical features,1


Index(['SETTLEMENT_PERIOD', 'Year', 'Month', 'Day', 'DayOfWeek'], dtype='object')


In [20]:
# Compare baseline models
best_model = compare_models(sort='RMSE', n_select=3)  # pick top 3 models

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.0427,0.0032,0.0560,0.9321,0.0049,0.0041,0.5980
et,Extra Trees Regressor,0.0437,0.0034,0.0581,0.9269,0.0051,0.0042,1.7120
rf,Random Forest Regressor,0.0437,0.0035,0.0592,0.9243,0.0052,0.0042,2.1920
xgboost,Extreme Gradient Boosting,0.0458,0.0037,0.0608,0.9196,0.0053,0.0044,0.4260
gbr,Gradient Boosting Regressor,0.0484,0.0040,0.0630,0.9140,0.0055,0.0046,1.3220
dt,Decision Tree Regressor,0.0469,0.0042,0.0643,0.9105,0.0056,0.0045,0.1480
knn,K Neighbors Regressor,0.0596,0.0064,0.0793,0.8633,0.0069,0.0057,0.1220
ada,AdaBoost Regressor,0.0823,0.0103,0.1009,0.7809,0.0088,0.0079,1.2180
ridge,Ridge Regression,0.1500,0.0343,0.1850,0.2614,0.0162,0.0144,0.3360
lar,Least Angle Regression,0.1500,0.0343,0.1850,0.2614,0.0162,0.0144,0.0700


In [24]:
# Create and tune the best model (example with top model from compare_models)
model = create_model(best_model[0])


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,0.0377,0.0025,0.0500,0.9396,0.0043,0.0036
1,0.0352,0.0023,0.0477,0.9470,0.0041,0.0034
2,0.0502,0.0041,0.0643,0.9127,0.0056,0.0048
3,0.0417,0.0032,0.0563,0.9365,0.0049,0.0040
4,0.0488,0.0038,0.0618,0.9248,0.0054,0.0047
Mean,0.0427,0.0032,0.0560,0.9321,0.0049,0.0041
Std,0.0059,0.0007,0.0064,0.0121,0.0006,0.0006


In [25]:
# Evaluate performance
evaluate_model(model)

# Predict on test set
preds_log = predict_model(model, data=test_df)
preds_log.head()

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Light Gradient Boosting Machine,0.2171,0.0595,0.2439,-0.1447,0.0217,0.0215


,SETTLEMENT_DATE,SETTLEMENT_PERIOD,TSD,ENGLAND_WALES_DEMAND,EMBEDDED_WIND_GENERATION,EMBEDDED_WIND_CAPACITY,EMBEDDED_SOLAR_GENERATION,EMBEDDED_SOLAR_CAPACITY,NON_BM_STOR,PUMP_STORAGE_PUMPING,...,ELECLINK_FLOW,VIKING_FLOW,GREENLINK_FLOW,Year,ND_log,Month,Day,DayOfWeek,log_demand,prediction_label
347672,2020-10-31,34,32587.0,29917.0,4465.0,6527.0,0.0,13322.0,0,4,...,0.0,0.0,0.0,2020,10.372991,10,31,5,10.372960,10.443637
347673,2020-10-31,35,34278.0,31449.0,4294.0,6527.0,0.0,13322.0,0,4,...,0.0,0.0,0.0,2020,10.424511,10,31,5,10.424481,10.480070
347674,2020-10-31,37,34168.0,31313.0,4024.0,6527.0,0.0,13322.0,0,3,...,0.0,0.0,0.0,2020,10.417867,10,31,5,10.417837,10.492844
347675,2020-10-31,48,23309.0,20556.0,3342.0,6527.0,0.0,13323.0,0,65,...,0.0,0.0,0.0,2020,9.994060,10,31,5,9.994014,10.153342
347676,2020-10-31,39,31901.0,29470.0,3901.0,6527.0,0.0,13322.0,0,5,...,0.0,0.0,0.0,2020,10.351277,10,31,5,10.351246,10.490693
